# CredResolve — Statistical Investigation & Counterfactual

**Purpose:** test the reported “recovery improved by 11% month-on-month” claim using defensible payment counts, inspect portfolio mix and data fitness, and specify how to estimate the impact of a targeting-strategy change.

**Scope caveat:** the supplied payment data runs from **1 January to 8 August 2026**, not a complete 12 months. August is partial. Results below describe the supplied synthetic challenge data and must not be presented as a verified real-world recovery rate.

**Definitions used here**
- **Gross successful-payment amount:** sum of canonical payment records whose status is `SUCCESS`; it is not net recovery and does not deduct reversals/refunds or collection cost.
- **Account payment coverage:** accounts with at least one canonical successful payment during the observed period divided by all accounts in the supplied account snapshot. It is not a recovery rate.
- **Canonical payment:** one record per `payment_id`, only after checking whether the core transaction fields agree across repeated IDs. A payment reference alone is not a safe deduplication key.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Find the project root without hard-coding a person's Windows path.
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "dataset" / "accounts.csv").exists() and (candidate / "dataset" / "payments.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find dataset/accounts.csv and dataset/payments.csv. "
        "Place this notebook under CredResolve/notebooks and keep the supplied CSVs in CredResolve/dataset."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = ["accounts", "payments", "borrowers", "agents", "campaigns", "daily_targeting", "account_status_history", "promises_to_pay"]
data = {name: pd.read_csv(DATA_DIR / f"{name}.csv", low_memory=False) for name in required}
accounts = data["accounts"].copy()
payments = data["payments"].copy()
borrowers = data["borrowers"].copy()
agents = data["agents"].copy()
campaigns = data["campaigns"].copy()
targeting = data["daily_targeting"].copy()
status_history = data["account_status_history"].copy()
ptp = data["promises_to_pay"].copy()

for frame, column in [
    (accounts, "opened_at"), (payments, "event_at"), (campaigns, "start_at"),
    (campaigns, "end_at"), (targeting, "target_date"),
    (status_history, "event_at"), (status_history, "recorded_at"),
    (ptp, "event_at"), (ptp, "promised_date")
]:
    frame[column] = pd.to_datetime(frame[column], errors="coerce")
payments["amount"] = pd.to_numeric(payments["amount"], errors="coerce")
print(f"Project root: {PROJECT_ROOT}")
print("Loaded row counts:")
display(pd.DataFrame({"table": list(data), "rows": [len(data[k]) for k in data]}))

Project root: C:\Users\GOPICHAND\Desktop\CredResolve
Loaded row counts:


,table,rows
0,accounts,30000
1,payments,25500
2,borrowers,30600
3,agents,30000
4,campaigns,120
5,daily_targeting,45000
6,account_status_history,60000
7,promises_to_pay,18000


## 1. Payment canonicalization and claim test

The core fields compared across repeated payment IDs are account, borrower, event timestamp, amount, status, payment method and provider. If those conflict, the ID should be quarantined for review; if they agree, choose one canonical row. `payment_reference` is not used alone because it is reused across different payment IDs.

In [2]:
core_fields = ["account_id", "borrower_id", "event_at", "amount", "payment_status", "payment_method", "provider_id"]
raw_payment_rows = len(payments)
raw_payment_ids = payments["payment_id"].nunique(dropna=True)
null_payment_ids = int(payments["payment_id"].isna().sum())
duplicate_id_groups = int((payments["payment_id"].value_counts(dropna=True) > 1).sum())

# Count per-ID variation across core transaction fields. dropna=False treats missing-vs-present as a difference.
core_versions = payments.dropna(subset=["payment_id"]).groupby("payment_id")[core_fields].nunique(dropna=False).max(axis=1)
conflicting_payment_ids = int((core_versions > 1).sum())
missing_id_rows = payments[payments["payment_id"].isna()].copy()
conflict_ids = set(core_versions[core_versions > 1].index)

# Mirror the documented SQL rule: prefer a non-null reference, then sort deterministically.
# Only IDs with consistent core fields are canonicalized; missing/conflicting IDs are excluded for review.
canonical_source = payments[payments["payment_id"].notna() & ~payments["payment_id"].isin(conflict_ids)].copy()
canonical_source["_reference_missing"] = canonical_source["payment_reference"].isna()
canonical_payments = (
    canonical_source.sort_values(["_reference_missing", "payment_reference"], na_position="last")
    .drop_duplicates("payment_id", keep="first")
    .drop(columns="_reference_missing")
)

successful = canonical_payments[canonical_payments["payment_status"].eq("SUCCESS")].copy()
successful["month"] = successful["event_at"].dt.to_period("M").astype(str)
monthly = successful.groupby("month", as_index=False).agg(
    successful_payment_count=("payment_id", "nunique"),
    gross_successful_amount=("amount", "sum"),
    first_payment_at=("event_at", "min"),
    last_payment_at=("event_at", "max")
)
monthly["days_observed"] = monthly.apply(lambda r: r["last_payment_at"].normalize().day - r["first_payment_at"].normalize().day + 1 if r["month"] == "2026-08" else pd.Period(r["month"]).days_in_month, axis=1)
monthly["mom_change_pct"] = monthly["gross_successful_amount"].pct_change() * 100
monthly["amount_per_observed_day"] = monthly["gross_successful_amount"] / monthly["days_observed"]
monthly.to_csv(OUTPUT_DIR / "statistical_monthly_payment_trend.csv", index=False)

summary = pd.DataFrame([
    ("Raw payment rows", raw_payment_rows),
    ("Distinct non-null payment IDs", raw_payment_ids),
    ("Duplicate payment-ID groups", duplicate_id_groups),
    ("Missing payment-ID rows", null_payment_ids),
    ("Payment IDs with conflicting core fields", conflicting_payment_ids),
    ("Canonical payment rows", len(canonical_payments)),
    ("Canonical successful payment rows", len(successful)),
    ("Canonical gross successful amount (₹)", round(successful["amount"].sum(), 2)),
], columns=["check", "result"])
display(summary)
display(monthly.round({"gross_successful_amount": 2, "mom_change_pct": 2, "amount_per_observed_day": 2}))

full_months = monthly[monthly["month"].between("2026-01", "2026-07")]
jan_amount = float(monthly.loc[monthly["month"].eq("2026-01"), "gross_successful_amount"].iloc[0])
jul_amount = float(monthly.loc[monthly["month"].eq("2026-07"), "gross_successful_amount"].iloc[0])
aug_per_day = float(monthly.loc[monthly["month"].eq("2026-08"), "amount_per_observed_day"].iloc[0])
jul_per_day = float(monthly.loc[monthly["month"].eq("2026-07"), "gross_successful_amount"].iloc[0] / 31)
print(f"Jan-to-Jul gross amount change: {(jul_amount / jan_amount - 1) * 100:.3f}%")
print(f"Aug 1–8 gross amount/day vs July gross amount/day: {(aug_per_day / jul_per_day - 1) * 100:.2f}% (descriptive only; August is partial).")

,check,result
0,Raw payment rows,2.550000e+04
1,Distinct non-null payment IDs,2.500000e+04
2,Duplicate payment-ID groups,5.000000e+02
3,Missing payment-ID rows,0.000000e+00
4,Payment IDs with conflicting core fields,0.000000e+00
5,Canonical payment rows,2.500000e+04
6,Canonical successful payment rows,1.753400e+04
7,Canonical gross successful amount (₹),1.315584e+09


,month,successful_payment_count,gross_successful_amount,first_payment_at,last_payment_at,days_observed,mom_change_pct,amount_per_observed_day
0,2026-01,2464,1.872291e+08,2026-01-01 00:14:40,2026-01-31 23:48:25,31,NaN,6039649.28
1,2026-02,2268,1.701425e+08,2026-02-01 01:10:28,2026-02-28 23:57:08,28,-9.13,6076516.21
2,2026-03,2524,1.889124e+08,2026-03-01 01:07:43,2026-03-31 23:41:43,31,11.03,6093947.55
3,2026-04,2406,1.751380e+08,2026-04-01 00:08:05,2026-04-30 23:45:47,30,-7.29,5837934.78
4,2026-05,2449,1.842503e+08,2026-05-01 00:01:39,2026-05-31 23:23:05,31,5.20,5943557.37
5,2026-06,2366,1.755597e+08,2026-06-01 00:14:23,2026-06-30 23:58:18,30,-4.72,5851990.90
6,2026-07,2441,1.872423e+08,2026-07-01 01:39:13,2026-07-31 23:39:36,31,6.65,6040073.07
7,2026-08,616,4.710970e+07,2026-08-01 00:11:28,2026-08-08 23:50:23,8,-74.84,5888711.91


Jan-to-Jul gross amount change: 0.007%
Aug 1–8 gross amount/day vs July gross amount/day: -2.51% (descriptive only; August is partial).


### Interpretation

- The canonical payment table contains 25,000 rows from 25,500 raw rows; 500 payment-ID groups repeat.
- Under the core-field checks above, the supplied data has no repeated payment IDs with conflicting core fields.
- Jan–Jul gross successful-payment amount is essentially flat from the first full month to the last. The monthly sequence is volatile, not a consistent +11% month-on-month trajectory.
- August's month-on-month amount comparison is not valid because only the first eight days are observed. A per-day comparison is shown only as a rough diagnostic, not as a full-month forecast.

This rejects the claim **as demonstrated by gross successful-payment amount**; it does not prove net recovery or establish a causal reason for the observed fluctuations.

## 2. Deduplication and attribution risks

In [3]:
# Payment-reference reuse: count references associated with more than one distinct payment ID.
ref_map = payments.dropna(subset=["payment_reference", "payment_id"]).groupby("payment_reference")["payment_id"].nunique()
reused_references = ref_map[ref_map > 1]
reused_reference_ids = payments[payments["payment_reference"].isin(reused_references.index)]["payment_id"].nunique()

# Campaign vs targeting dates. Date-level alignment is a diagnostic; exact timestamps and time zones need normalization.
campaign_window = campaigns[["campaign_id", "strategy_version", "start_at", "end_at"]].copy()
target_campaign = targeting.merge(campaign_window, on="campaign_id", how="left", validate="many_to_one")
target_day = target_campaign["target_date"].dt.normalize()
start_day = target_campaign["start_at"].dt.normalize()
end_day = target_campaign["end_at"].dt.normalize()
before_start = int((target_day < start_day).sum())
after_end = int((target_day > end_day).sum())
in_window = int(((target_day >= start_day) & (target_day <= end_day)).sum())

attribution_checks = pd.DataFrame([
    ("Payment references reused across >1 payment ID", len(reused_references)),
    ("Payment IDs appearing under reused references", reused_reference_ids),
    ("Targeting rows", len(target_campaign)),
    ("Targeting rows before campaign start date", before_start),
    ("Targeting rows after campaign end date", after_end),
    ("Targeting rows within campaign start/end dates", in_window),
], columns=["check", "result"])
display(attribution_checks)

campaigns["start_month"] = campaigns["start_at"].dt.to_period("M").astype(str)
version_by_start_month = pd.crosstab(campaigns["start_month"], campaigns["strategy_version"])
display(version_by_start_month)
version_by_target_month = pd.crosstab(target_campaign["target_date"].dt.to_period("M").astype(str), target_campaign["strategy_version"])
display(version_by_target_month)
version_by_target_month.to_csv(OUTPUT_DIR / "target_strategy_version_by_month.csv")

,check,result
0,Payment references reused across >1 payment ID,3407
1,Payment IDs appearing under reused references,7218
2,Targeting rows,45000
3,Targeting rows before campaign start date,15253
4,Targeting rows after campaign end date,20886
5,Targeting rows within campaign start/end dates,8861


strategy_version,legacy,v1,v2,v3
start_month,,,,
2026-01,10,8,1,5
2026-02,4,3,8,4
2026-03,6,2,9,10
2026-04,11,6,2,11
2026-05,6,5,7,2


strategy_version,legacy,v1,v2,v3
target_date,,,,
2026-01,2002,1298,1380,1689
2026-02,1743,1150,1243,1573
2026-03,1922,1259,1408,1701
2026-04,1906,1280,1425,1594
2026-05,1978,1293,1494,1677
2026-06,1929,1224,1397,1604
2026-07,1922,1266,1416,1626
2026-08,503,340,355,403


### Interpretation

- **Payment references must not be used as a unique transaction key.** In the source data, 3,407 references are associated with multiple distinct payment IDs, covering 7,218 payment IDs.
- Of 45,000 targeting rows, only 8,861 align with campaign start/end **calendar dates**; 15,253 are dated before the campaign starts and 20,886 after it ends. This is a high-priority campaign-attribution defect. Because timestamps and time zones are not yet normalized, treat these as date-level flags to investigate, not final proof that every row is invalid.
- Strategy labels (`legacy`, `v1`, `v2`, `v3`) are interleaved across campaign-start months and targeting months. They do not establish one clean global cutover date. The actual strategy change date must come from an approved change log or a reliable assignment record before a retrospective causal estimate can be claimed.

## 3. Portfolio mix and cohort checks

The outcomes below use the **current account snapshot** joined to successful payments over the observed period. They are descriptive account payment coverage and gross amounts, not historical recovery rates. Risk segment, DPD and status may have changed since payment events; they must not be treated as time-aligned historical attributes.

In [4]:
payment_by_account = successful.groupby("account_id", as_index=False).agg(
    successful_payments=("payment_id", "nunique"),
    gross_successful_amount=("amount", "sum")
)
portfolio = accounts.copy()
portfolio["dpd"] = pd.to_numeric(portfolio["dpd"], errors="coerce")
portfolio["principal_amount"] = pd.to_numeric(portfolio["principal_amount"], errors="coerce")
portfolio["outstanding_amount"] = pd.to_numeric(portfolio["outstanding_amount"], errors="coerce")
portfolio["opened_at"] = pd.to_datetime(portfolio["opened_at"], errors="coerce")
portfolio["dpd_bucket"] = pd.cut(
    portfolio["dpd"], bins=[-1, 0, 30, 60, 90, 180, np.inf],
    labels=["0", "1–30", "31–60", "61–90", "91–180", "181+"]
)
portfolio["vintage_year"] = portfolio["opened_at"].dt.year.astype("Int64").astype(str)
portfolio = portfolio.merge(payment_by_account, on="account_id", how="left", validate="one_to_one")
portfolio["successful_payments"] = portfolio["successful_payments"].fillna(0)
portfolio["gross_successful_amount"] = portfolio["gross_successful_amount"].fillna(0.0)
portfolio["has_successful_payment"] = portfolio["successful_payments"].gt(0)

def group_coverage(frame, key):
    out = frame.groupby(key, dropna=False, observed=False).agg(
        accounts=("account_id", "nunique"),
        accounts_with_success=("has_successful_payment", "sum"),
        gross_successful_amount=("gross_successful_amount", "sum"),
    ).reset_index()
    out["account_payment_coverage_pct"] = 100 * out["accounts_with_success"] / out["accounts"]
    return out

risk_mix = group_coverage(portfolio, "risk_segment")
dpd_mix = group_coverage(portfolio, "dpd_bucket")
loan_mix = group_coverage(portfolio, "loan_type")
vintage_mix = group_coverage(portfolio, "vintage_year")
for title, frame in [("Risk segment", risk_mix), ("DPD bucket", dpd_mix), ("Loan type", loan_mix), ("Account opening year", vintage_mix)]:
    print(title)
    display(frame.round({"gross_successful_amount": 2, "account_payment_coverage_pct": 2}))
    frame.to_csv(OUTPUT_DIR / (title.lower().replace(" ", "_") + "_coverage.csv"), index=False)

# Check the snapshot-quality caveat flagged in prior validation.
closed_like_positive = int((portfolio["status"].isin(["CLOSED", "PAID", "WRITEOFF"]) & portfolio["outstanding_amount"].gt(0)).sum())
outstanding_gt_principal = int((portfolio["outstanding_amount"] > portfolio["principal_amount"]).sum())
print(f"Closed/paid/writeoff accounts with positive reported outstanding: {closed_like_positive:,}")
print(f"Accounts whose reported outstanding exceeds principal: {outstanding_gt_principal:,}")

Risk segment


,risk_segment,accounts,accounts_with_success,gross_successful_amount,account_payment_coverage_pct
0,HIGH,7552,3300,3.315982e+08,43.70
1,LOW,7513,3381,3.308578e+08,45.00
2,MEDIUM,7533,3335,3.305101e+08,44.27
3,NPA,7402,3268,3.226178e+08,44.15


DPD bucket


,dpd_bucket,accounts,accounts_with_success,gross_successful_amount,account_payment_coverage_pct
0,0,2685,1220,1.174619e+08,45.44
1,1–30,10880,4719,4.715398e+08,43.37
2,31–60,5514,2524,2.527771e+08,45.77
3,61–90,5468,2436,2.387357e+08,44.55
4,91–180,5453,2385,2.350695e+08,43.74
5,181+,0,0,0.000000e+00,NaN


Loan type


,loan_type,accounts,accounts_with_success,gross_successful_amount,account_payment_coverage_pct
0,AUTO,6079,2700,2.679332e+08,44.42
1,BNPL,5928,2580,2.554400e+08,43.52
2,CONSUMER,5930,2685,2.669767e+08,45.28
3,CREDIT_CARD,6080,2690,2.686432e+08,44.24
4,PERSONAL,5983,2629,2.565909e+08,43.94


Account opening year


,vintage_year,accounts,accounts_with_success,gross_successful_amount,account_payment_coverage_pct
0,2024,15747,6986,6.910171e+08,44.36
1,2025,14253,6298,6.245668e+08,44.19


Closed/paid/writeoff accounts with positive reported outstanding: 22,461
Accounts whose reported outstanding exceeds principal: 13,029


### Interpretation

- Current-snapshot account payment coverage is similar across the broad risk groups (about 43.7%–45.0%), DPD buckets (about 43.4%–45.8%) and loan types (about 43.5%–45.3%). These unadjusted differences do **not** identify why payments occurred and should not be used to rank channel or targeting effectiveness.
- Account-opening-year coverage is also very similar (about 44.2%–44.4%). This is a simple vintage check, not a full cohort model.
- The source snapshot has material balance/status contradictions: 22,461 accounts marked `CLOSED`, `PAID` or `WRITEOFF` still have positive outstanding, and 13,029 show outstanding greater than principal. These may reflect fees, stale status, definitions or errors; they require reconciliation before calculating a financial recovery rate.
- Geography is not used for conclusions here because the borrower table has severe duplicate/conflicting borrower identities and unmatched account-to-borrower links. The input also has no clear client or language dimension, so client/language effects cannot be tested from the supplied columns.

## 4. Cohort, identity, time-zone and denominator risks

In [5]:
# Entity-resolution diagnostics.
agent_by_id = agents.groupby("agent_id").agg(
    rows=("agent_id", "size"),
    employee_codes=("employee_code", "nunique"),
    names=("agent_name", "nunique"),
    vendors=("vendor_id", "nunique"),
    joined_dates=("joined_at", "nunique"),
    statuses=("status", "nunique"),
)
borrower_by_id = borrowers.groupby("borrower_id").agg(
    rows=("borrower_id", "size"),
    names=("name", "nunique"),
    phones=("phone", "nunique"),
    emails=("email", "nunique"),
    cities=("city", "nunique"),
)
account_borrower_ids = set(accounts["borrower_id"].dropna())
known_borrower_ids = set(borrowers["borrower_id"].dropna())
missing_account_borrower = int(accounts["borrower_id"].isna().sum())
unmatched_nonnull_borrower = int((accounts["borrower_id"].notna() & ~accounts["borrower_id"].isin(known_borrower_ids)).sum())

identity_summary = pd.DataFrame([
    ("Agent source rows", len(agents)),
    ("Distinct agent IDs", agents["agent_id"].nunique()),
    ("Agent IDs with >1 employee code", int((agent_by_id["employee_codes"] > 1).sum())),
    ("Agent IDs with >1 name", int((agent_by_id["names"] > 1).sum())),
    ("Agent IDs with >1 vendor", int((agent_by_id["vendors"] > 1).sum())),
    ("Borrower source rows", len(borrowers)),
    ("Distinct borrower IDs", borrowers["borrower_id"].nunique()),
    ("Borrower IDs with >1 name", int((borrower_by_id["names"] > 1).sum())),
    ("Borrower IDs with >1 phone", int((borrower_by_id["phones"] > 1).sum())),
    ("Accounts missing borrower ID", missing_account_borrower),
    ("Accounts with non-null unmatched borrower ID", unmatched_nonnull_borrower),
], columns=["check", "result"])
display(identity_summary)
identity_summary.to_csv(OUTPUT_DIR / "entity_resolution_diagnostics.csv", index=False)

# Treat these timestamp differences as diagnostic only: timezone columns are inconsistent/not present on every table.
status_history["lag_hours_naive"] = (status_history["recorded_at"] - status_history["event_at"]).dt.total_seconds() / 3600
late_diagnostic = pd.DataFrame([
    ("Status-history rows", len(status_history)),
    ("Rows with recorded_at earlier than event_at under naive comparison", int((status_history["lag_hours_naive"] < 0).sum())),
    ("Rows with absolute naive lag > 6 hours", int((status_history["lag_hours_naive"].abs() > 6).sum())),
    ("Promises with promised_date after 2026-08-08 cutoff", int((ptp["promised_date"] > pd.Timestamp("2026-08-08 23:59:59")).sum())),
], columns=["check", "result"])
display(late_diagnostic)
late_diagnostic.to_csv(OUTPUT_DIR / "timestamp_and_censoring_diagnostics.csv", index=False)

,check,result
0,Agent source rows,30000
1,Distinct agent IDs,1000
2,Agent IDs with >1 employee code,1000
3,Agent IDs with >1 name,1000
4,Agent IDs with >1 vendor,1000
5,Borrower source rows,30600
6,Distinct borrower IDs,11015
7,Borrower IDs with >1 name,8185
8,Borrower IDs with >1 phone,8399
9,Accounts missing borrower ID,455


,check,result
0,Status-history rows,60000
1,Rows with recorded_at earlier than event_at un...,30191
2,Rows with absolute naive lag > 6 hours,45107
3,Promises with promised_date after 2026-08-08 c...,1242


### Statistical interpretation by evidence class

| Classification | Conclusion |
|---|---|
| **Fact** | After canonicalizing repeated payment IDs with consistent core fields, there are 17,534 successful payment IDs totaling ₹1,315,583,964.64 in gross successful payments. |
| **Strong evidence** | The supplied monthly gross-payment series does not show sustained +11% month-on-month growth; January and July are almost identical. |
| **Fact** | Campaign targeting dates do not align with campaign active dates for most target rows under the date-level check; payment references and entity IDs are not reliable unique identifiers. |
| **Correlation only** | Current-snapshot risk/DPD/vintage groups have slightly different account payment coverage. These are unadjusted associations, not channel or strategy effects. |
| **Hypothesis** | The reported +11% could be driven by changing denominators, duplicate events, attribution rules, portfolio mix, timezone mapping or campaign-version mapping. These remain hypotheses until the metric definition and historical eligible population are reconciled. |

A formal Simpson's-paradox test is not supportable yet: the current DPD/risk/status fields are snapshots, not reliable monthly as-of values. A valid test requires monthly eligible-account denominators and the same outcome window for each stratum.

## 5. Counterfactual: what if targeting had not changed?

### Preferred retrospective design: matched difference-in-differences

**Do not infer treatment solely from the `strategy_version` label.** Campaign versions appear in overlapping months, and target assignments fall outside campaign calendar windows. Obtain the approved targeting-change effective date and an immutable record of which eligible accounts were assigned the new vs legacy policy.

1. **Treatment group:** accounts eligible at the cutover date that were assigned the new targeting policy.
2. **Control group:** eligible accounts contemporaneously retained on the legacy policy (ideally a randomized holdout; otherwise matched/weighted accounts with common support).
3. **Time zero:** actual assignment timestamp, normalized to a canonical time zone.
4. **Primary outcome:** cumulative **net** collected amount per eligible assigned account in the 30 days after time zero, including zero for accounts with no payment. Deduplicate transaction IDs; correctly link reversals/refunds to original payments. Until those links are reliable, report gross successful payments as a secondary proxy only.
5. **Pre-period:** the comparable 30-day period before assignment; also inspect 60/90-day prior payment, contact and PTP history.
6. **Baseline covariates:** pre-treatment DPD, risk, outstanding balance, loan type, account vintage, previous payments, previous contact/PTP history and geography if entity resolution is repaired. Use agent/vendor/channel variables only after identity and timezone validation.
7. **Estimator:**

   `DiD = (Treatment_post − Treatment_pre) − (Control_post − Control_pre)`

   For amount outcomes, estimate at account level and cluster uncertainty at borrower level if borrower identity can be trusted. Standardize segment rates to a fixed pre-change portfolio mix to separate operational change from mix shift.
8. **Assumptions:** parallel pre-trends, stable outcome definitions, comparable follow-up windows, no unmeasured differential interventions, no spillovers between groups, and no treatment-dependent loss of accounts from the denominator.
9. **Diagnostics:** plot pre-trends; run placebo cutover dates; test alternative 7/30/60-day windows; compare raw, matched and weighted estimates; show confidence intervals and effect by pre-specified DPD/risk strata; retain an event-study plot.
10. **Limitations:** only about eight months are present; campaign mapping and identities are inconsistent; timestamps require normalization; payment reversals are not reliably paired. If a credible concurrent control and exact cutover date cannot be recovered, a retrospective causal estimate is not defensible.

### Better follow-up: randomized controlled pilot

Randomly assign eligible accounts (or borrower clusters after identity resolution) to new targeting vs business-as-usual. Stratify randomization by pre-treatment DPD, risk, balance and loan type. Preserve a pre-registered holdout sized by power analysis, keep all assigned accounts in the intention-to-treat denominator, use a fixed 30-day outcome window, and track net collections, cost, complaints and operational capacity. Scale only when incremental net recovery and ROI are positive with an acceptable uncertainty range.

## 6. What is and is not established

**Established from the supplied data:** duplicate payment-ID counts and canonical totals; volatile/flat gross payment trends; identity-resolution defects; campaign-window inconsistencies; current-snapshot account coverage; and the incomplete observed period.

**Not established:** true net recovery rate; the cause of each monthly change; causal channel effectiveness; causal impact of targeting changes; cost per rupee recovered; ROI for a ₹10 crore investment; reliable agent-tenure effects; or reliable client/language effects.

**Next data needed:** approved metric definitions; the strategy-change log and assignment audit trail; historical daily eligible-account snapshots; canonical entity crosswalks; timezone/source-offset rules; reversal/refund-to-original-payment linkage; consistent campaign active windows; cost and agent-hour data; and reliable agent/client/language dimensions.